# 실습 4 - AutoGluon 시계열 모델 학습과 예측

**서울 공공자전거(따릉이) 대여량 14일 예측**

- 앞선 실습(실습 3): 데이터를 시계열 형식으로 준비함
- 본 실습: 준비된 데이터로 **모델을 학습**하고 **미래를 예측**함

#### 본 실습에서 확인할 사항
1. 학습·평가 분할은 **시간 순**으로 수행함 (무작위 분할 불가)
2. `fit()` 한 줄로 **여러 모델이 자동 학습**됨 (통계·트리·딥러닝·사전학습)
3. 예측 결과는 **값 하나가 아니라 범위**로 나옴 (확률적 예측)
4. 어떤 변수가 중요했는지 확인 가능함

#### 정형 데이터 실습과 같은 흐름
- `fit()` → `leaderboard()` → `predict()` → `feature_importance()`
- 명령 이름과 순서가 동일함

#### 노트북 사용법
- 코드 셀 선택 후 **Shift + Enter** 로 실행
- 상단에서부터 순서대로 실행

In [ ]:
# !pip install -U pip
# !pip install -U setuptools wheel
# !pip install autogluon
# !pip install -U accelerate transformers
# !pip install -U bitsandbytes
# !pip install -U ipywidgets

---
## 0. 환경 준비

- `pandas` : 표 형태 데이터 처리
- `TimeSeriesDataFrame` : 시계열 데이터 형식
- `TimeSeriesPredictor` : 시계열 모델 학습·예측

In [ ]:
import os
import pandas as pd
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

os.environ["LOKY_MAX_CPU_COUNT"] = "16"
os.environ["UCX_TLS"] = "tcp,sm,self"
os.environ["OMPI_MCA_btl"] = "^openib"
os.environ["OMPI_MCA_pml"] = "ob1"

print("준비 완료")

---
## 1. 데이터 준비

- 실습 3의 내용을 요약하여 재현함
- 시간별 → 일별 집계 → 세 개의 열 + 추가 변수 → `TimeSeriesDataFrame`
- 집계 방식은 변수마다 다름 (합계 / 평균 / 첫 값)

In [ ]:
df = pd.read_csv("../SeoulBikeData.csv", encoding="latin-1")
df["Date"] = pd.to_datetime(df["Date"], format="%d/%m/%Y")

# 일별 집계 — 변수마다 다른 방식
daily = (
    df.groupby("Date")
    .agg(
        {
            "Rented Bike Count": "sum",  # 합계
            "Rainfall(mm)": "sum",
            "Snowfall (cm)": "sum",
            "Solar Radiation (MJ/m2)": "sum",
            "Temperature(°C)": "mean",  # 평균
            "Humidity(%)": "mean",
            "Wind speed (m/s)": "mean",
            "Visibility (10m)": "mean",
            "Dew point temperature(°C)": "mean",
            "Seasons": "first",  # 첫 값
            "Holiday": "first",
            "Functioning Day": "first",
        }
    )
    .round(2)
    .reset_index()
)

# 열 이름 정리
ts_df = daily.rename(
    columns={
        "Date": "timestamp",
        "Rented Bike Count": "target",
        "Temperature(°C)": "temperature",
        "Humidity(%)": "humidity",
        "Wind speed (m/s)": "wind_speed",
        "Visibility (10m)": "visibility",
        "Dew point temperature(°C)": "dew_point",
        "Solar Radiation (MJ/m2)": "solar_radiation",
        "Rainfall(mm)": "rainfall",
        "Snowfall (cm)": "snowfall",
        "Seasons": "seasons",
        "Holiday": "holiday",
        "Functioning Day": "functioning_day",
    }
)
ts_df["item_id"] = "seoul_bike"

ts_data = TimeSeriesDataFrame.from_data_frame(
    ts_df,
    id_column="item_id",
    timestamp_column="timestamp",
)

print("크기:", ts_data.shape, "| 시간 간격:", ts_data.freq)
ts_data.head()

#### 확인 사항
- 365행 (1년치 일별 데이터)
- 시간 간격 = `D` (일별)
- `target`(대여량) + 추가 변수 12개

---
## 2. 학습·평가 분할 (시간 순)

- 정형 데이터: 무작위로 섞어 분할
- 시계열 데이터: **시간 순으로 분할** — 무작위 분할 불가
  - 무작위로 섞으면 미래 데이터로 과거를 예측하는 셈이 됨
  - 실제 상황과 다름 → 성능이 실제보다 부풀려짐

#### 두 개의 숫자를 구분함
- **예측 구간(`prediction_length`) = 14일** : 며칠 앞을 예측할 것인가 → **연구 목적**이 정함
- **평가용 크기 = 14일 × 3창 = 42일** : 몇 번 시험 볼 것인가 → **평가 신뢰도**가 정함

> 14일 한 번만 채점하면 그 2주의 우연(폭우·이상기온)에 성적이 좌우됨
> → 시점을 옮겨 3번 시험하면 신뢰할 수 있는 성능 추정 가능

In [ ]:
prediction_length = 14  # 며칠 앞을 예측할 것인가
num_test_windows = 3  # 몇 번 시험 볼 것인가

train_data, test_data = ts_data.train_test_split(
    num_test_windows * prediction_length  # 마지막 42일을 평가용으로
)

print("전체  시점 개수:", ts_data.num_timesteps_per_item().values)
print("학습용 시점 개수:", train_data.num_timesteps_per_item().values)
print("평가용 시점 개수:", test_data.num_timesteps_per_item().values)

#### 결과 해석
- **학습용(train_data)** : 마지막 42일을 제외한 과거 데이터 (323일)
- **평가용(test_data)** : 전체 데이터 (365일)
  - 과거 + 미래 42일을 모두 포함함
  - 채점 시에만 마지막 42일을 사용함
  - 앞의 323일은 모델이 "지금까지의 흐름"을 파악하는 데 필요함
- 과거로만 학습하고 잘라낸 미래로 시험 → **실제 예측 상황과 동일한 조건**

---
## 3. 모델 학습 `fit()`

- 정형 데이터의 `TabularPredictor`와 **같은 구조**
- `prediction_length` : 며칠 앞을 예측할 것인가
- `target` : 예측 대상 열
- `freq` : 시간 간격 (지정 시 빠진 시점을 자동으로 채움)
- 나머지 열(기온·습도 등)은 **자동으로 보조 변수로 인식됨**

In [ ]:
predictor = TimeSeriesPredictor(
    prediction_length=prediction_length,  # 14일 앞 예측
    # known_covariates_names=["seasons", "holiday"],
    target="target",  # 예측 대상 = 대여량
    freq="D",  # 일별 데이터
).fit(
    train_data,
)

#### 학습 로그에서 확인할 사항

1. **`Frequency of time series data: 'D'`**
   → 일별 데이터로 인식됨

2. **`Training timeseries model ...`** 의 반복
   → 여러 모델이 순차적으로 학습됨
   - `SeasonalNaive`, `AutoETS` … → **통계 모델**
   - `RecursiveTabular`, `DirectTabular` → **트리 기반** (LightGBM 사용)
   - `TemporalFusionTransformer`, `DeepAR` … → **딥러닝**
   - `Chronos` → **사전학습 모델**

3. **`Training timeseries model WeightedEnsemble`**
   → 개별 모델들을 결합한 **앙상블** 생성

4. **`Best model: WeightedEnsemble`**
   → 최종 선택된 모델

> 정형 데이터와 동일한 리듬 — `fit()` 한 줄에 여러 모델 학습 + 앙상블

---
## 4. 성능 비교 `leaderboard()`

- 학습된 모델들의 성능을 비교함
- `leaderboard(test_data)` : 평가용 데이터로 각 모델을 채점하여 순위표 출력

In [ ]:
predictor.leaderboard(test_data)

#### 결과 해석
- **`model`** : 모델 이름
- **`score_test`** : 평가용 데이터에 대한 성능
  - **0에 가까울수록 우수함** (음수의 절댓값이 작을수록 우수)
  - AutoGluon은 '클수록 우수함'으로 부호를 통일함 → **최상단 행이 가장 우수한 모델**
- **`fit_time_marginal`** : 해당 모델의 학습 소요 시간(초)

---
## 5. 예측 `predict()`

- 완성된 모델로 미래 14일의 대여량을 예측함
- 가장 우수한 모델(앙상블)이 자동으로 사용됨

In [ ]:
# past_data, known_covariates = test_data.get_model_inputs_for_scoring(
#     prediction_length=prediction_length,
#     known_covariates_names=["seasons", "holiday"],
# )

# predictions = predictor.predict(past_data, known_covariates=known_covariates)
predictions = predictor.predict(past_data, model="Chronos2")
predictions

#### 결과 해석 — 값 하나가 아니라 **범위**

- 정형 데이터의 회귀: 예측값 **하나**
- 시계열 예측: **평균 + 분위수**를 함께 출력함 → **확률적 예측**

| 열 | 의미 |
|---|---|
| `mean` | 평균 예측 (기댓값) |
| `0.1` | P10 — 10% 확률로 이 값보다 낮음 |
| `0.5` | P50 — 중앙값 |
| `0.9` | P90 — 90% 확률로 이 값보다 낮음 |

- **P10 ~ P90 사이에 있을 확률 = 80%**
- 예: "내일 대여량은 평균 2만 대, 80% 확률로 1만7천 ~ 2만3천 대"
- 단일 값보다 **의사결정에 유용함** (자전거를 몇 대 배치할 것인가)

> 미래로 갈수록 범위가 **넓어짐** → 멀수록 불확실함

---
## 6. 예측 시각화 `plot()`

- 과거 흐름 + 예측 + 실제값을 한 그래프로 확인함
- `quantile_levels` : 표시할 분위수 범위
- `max_history_length` : 과거를 며칠까지 보여줄 것인가

In [ ]:
predictor.plot(
    data=test_data,
    predictions=predictions,
    quantile_levels=[0.1, 0.9],  # P10 ~ P90 범위 표시
    max_history_length=60,  # 과거 60일만 표시
);

#### 그래프 읽는 법
- **파란 선** : 과거 관측값 + 실제값
- **주황 선** : 평균 예측
- **음영 영역** : P10 ~ P90 (80% 예측 범위)
- 음영이 **오른쪽으로 갈수록 넓어짐** → 미래로 갈수록 불확실

#### 확인할 점
- 예측이 실제값의 흐름을 따라가는가?
- 실제값이 음영 범위 **안**에 들어오는가?
- 범위를 벗어난 날이 있다면 → **그날 무슨 일이 있었는가?**

---
## 7. 연구자가 조절하는 설정

- AutoGluon은 개별 모델의 세부 하이퍼파라미터를 연구자가 조정할 필요가 없도록 설계됨
- 대신 연구자는 다음 **상위 설정**으로 학습의 방향을 결정함

| 설정 | 역할 | 값의 예 |
|---|---|---|
| `prediction_length` | 며칠 앞을 예측할 것인가 | `14` (2주) |
| `presets` | 성능과 학습 시간의 균형 | `"fast_training"` / `"medium_quality"` / `"high_quality"` / `"best_quality"` |
| `time_limit` | 학습에 사용할 최대 시간(초) | `600` (10분) |
| `eval_metric` | 성능 판단 기준 | `"MASE"`, `"RMSE"`, `"WQL"` 등 |

- `prediction_length`와 `eval_metric`은 **연구 목적에 따라 결정됨**
- 무엇을 정확히 예측해야 하는가는 기계가 아니라 연구자가 판단하는 영역임

#### 예시: 평가지표(`eval_metric`) 변경 후 학습

- 기본값 대신 `RMSE`를 평가지표로 지정하여 재학습함
- 학습 시간 단축을 위해 `time_limit`을 짧게 지정함

In [ ]:
predictor_rmse = TimeSeriesPredictor(
    prediction_length=prediction_length,
    target="target",
    freq="D",
    eval_metric="RMSE",
).fit(
    train_data,
    presets="fast_training", 
    time_limit=180,
)

predictor_rmse.leaderboard(test_data)

#### 결과 해석
- `eval_metric`이 `RMSE`로 변경됨
- 평가 기준이 달라지면 **모델의 순위나 최적 모델이 달라질 수 있음**
- 즉, 어떤 기준으로 우수한 모델을 선정하는가에 따라 결과가 달라짐

> **참고: 대표적인 평가지표**
> - `RMSE` : 큰 오차에 더 큰 벌점 → 큰 실수를 피하고 싶을 때
> - `MASE` : 단순 예측(SeasonalNaive) 대비 얼마나 나은지 → 1보다 작으면 개선
> - `WQL` : 확률적 예측(분위수)의 품질까지 평가

---
## 8. 변수 중요도 `feature_importance()`

- 어떤 변수가 예측에 큰 영향을 미쳤는지 확인함
- **정형 데이터와 같은 원리** — 순열 중요도(permutation importance)
  - 어떤 변수의 값을 무작위로 뒤섞음 → 성능이 얼마나 떨어지는가 측정
  - 크게 떨어질수록 그 변수가 중요함

> 계산에 시간이 소요됨

In [ ]:
# predictor.feature_importance(test_data, model="Chronos2")
predictor.feature_importance(test_data)

#### 결과 해석
- `importance` 값이 **클수록 해당 변수가 예측에 중요**함
- 음수이면 → 오히려 예측을 방해한 변수 (제거 검토 대상)

#### 확인할 점
- 기온·강수량·적설량 등 **날씨 변수**가 상위에 오르는가?
- 대여량은 날씨에 크게 좌우되므로, 날씨 변수의 중요도가 높을 것으로 예상됨

> **왜 날씨 변수를 넣었는가**
> - 과거 대여량만으로 예측하면 **갑작스러운 날씨 변화를 포착할 수 없음**
> - 예: 2018년 11월 24일 — 폭설로 대여량이 6,477대까지 급락
>   (전날 16,314대 → 60% 감소)
> - 과거 흐름만 본 모델은 이런 날을 맞힐 수 없음
> - 관련 변수를 함께 넣어야 이러한 급변에 대응 가능함

---
## 정리

| 단계 | 명령 | 내용 |
|---|---|---|
| 분할 | `train_test_split(42)` | 시간 순 · 마지막 42일을 평가용으로 |
| 학습 | `TimeSeriesPredictor(...).fit()` | 여러 모델 자동 학습 + 앙상블 |
| 성능 | `predictor.leaderboard(test_data)` | 모델별 성능 비교 (최상단이 최우수) |
| 예측 | `predictor.predict(train_data)` | 평균 + 분위수(P10~P90) |
| 시각화 | `predictor.plot(...)` | 과거·예측·범위를 그래프로 |
| 설정 조절 | `prediction_length`, `presets`, `time_limit`, `eval_metric` | 연구자가 지정 |
| 해석 | `predictor.feature_importance(...)` | 변수별 중요도 |

**요약**
- 정형 데이터와 **같은 흐름** — `fit()` → `leaderboard()` → `predict()` → `feature_importance()`
- 차이점 세 가지
  1. 분할이 **시간 순** (무작위 불가)
  2. 예측이 **범위**로 나옴 (평균 + P10~P90)
  3. 시계열 전용 모델 사용 (통계·트리·딥러닝·사전학습)